<a href="https://colab.research.google.com/github/123bksri/DBA_Walsh_Capstone_Pneumonia_detection/blob/main/16th_Deadline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import os
import re
import tensorflow as tf
from   tensorflow.keras import layers, ops

print("TensorFlow version:", tf.__version__)
print("Keras version:",      tf.keras.__version__)

TensorFlow version: 2.19.0
Keras version: 3.13.2


Task-1

In [16]:
import urllib.request
url = "https://www.gutenberg.org/files/345/345-0.txt"
urllib.request.urlretrieve(url, "Dracula.txt")
print("Dracula.txt downloaded successfully!")

Dracula.txt downloaded successfully!


In [17]:
# Read the file
with open('Dracula.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

print("=== First ~1000 characters BEFORE preprocessing ===")
print(raw_text[:1000])

# Remove non-printable characters (keep only printable ASCII)
cleaned_text = re.sub(r'[^\x20-\x7E\n\t]', '', raw_text)

print("\n=== First ~1000 characters AFTER preprocessing ===")
print(cleaned_text[:1000])
print(f"\nTotal characters after cleaning: {len(cleaned_text)}")

=== First ~1000 characters BEFORE preprocessing ===
*** START OF THE PROJECT GUTENBERG EBOOK 345 ***




                                DRACULA

                                  _by_

                              Bram Stoker

                        [Illustration: colophon]

                                NEW YORK

                            GROSSET & DUNLAP

                              _Publishers_

      Copyright, 1897, in the United States of America, according
                   to Act of Congress, by Bram Stoker

                        [_All rights reserved._]

                      PRINTED IN THE UNITED STATES
                                   AT
               THE COUNTRY LIFE PRESS, GARDEN CITY, N.Y.




                                   TO

                             MY DEAR FRIEND

                               HOMMY-BEG




Contents

CHAPTER I. Jonathan Harker’s Journal
CHAPTER II. Jonathan Harker’s Journal
CHAPTER III. Jonathan Harker’s Journal
CHAPTER IV. Jon

Task-2

In [18]:
# Extract all unique characters and sort them
unique_chars = sorted(set(cleaned_text))
print(f"Number of unique characters: {len(unique_chars)}")
print(f"Unique characters: {unique_chars}")

# Create char-to-int and int-to-char dictionaries
char_to_int = {ch: idx for idx, ch in enumerate(unique_chars)}
int_to_char = {idx: ch for idx, ch in enumerate(unique_chars)}

# Encode function: string -> numpy array of ints
def encode(string):
    return np.array([char_to_int[ch] for ch in string], dtype=np.int32)

# Decode function: numpy array of ints -> string
def decode(array):
    return ''.join([int_to_char[idx] for idx in array])

# Test with 'vampire'
test_word = 'vampire'
encoded = encode(test_word)
decoded = decode(encoded)
print(f"\nEncoding of '{test_word}': {encoded}")
print(f"Decoded back: '{decoded}'")

Number of unique characters: 80
Unique characters: ['\n', ' ', '!', '&', '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}']

Encoding of 'vampire': [73 52 64 67 60 69 56]
Decoded back: 'vampire'


Task-3

In [19]:
batchSize  = 64    # integer, minibatch size
dimModel   = 128   # integer, model dimensionality (char embeddings and intermediate vectors)
epochs     = 10     # integer (at least 2 for final run)
ffDim      = 256   # integer, neurons of feedforward net within transformer (2x dimModel)
maxLength  = 129   # integer, must be >= windowSize + 1
numHeads   = 4     # integer, number of attention heads per transformer block (must divide dimModel evenly)
numLayers  = 2     # integer, defines number of transformer blocks to be built by the language model
vocabSize  = len(unique_chars)  # integer, the number of unique characters
windowSize = 128   # integer, largest context length for the character model

print(f"batchSize  = {batchSize}")
print(f"dimModel   = {dimModel}")
print(f"epochs     = {epochs}")
print(f"ffDim      = {ffDim}")
print(f"maxLength  = {maxLength}")
print(f"numHeads   = {numHeads}")
print(f"numLayers  = {numLayers}")
print(f"vocabSize  = {vocabSize}")
print(f"windowSize = {windowSize}")

batchSize  = 64
dimModel   = 128
epochs     = 10
ffDim      = 256
maxLength  = 129
numHeads   = 4
numLayers  = 2
vocabSize  = 80
windowSize = 128


Task-4

In [20]:
# Encode the entire cleaned text
encoded_text = encode(cleaned_text)

# Create inputs and targets arrays
# Total number of sequences
num_sequences = len(encoded_text) - windowSize

# Build inputs: each row is windowSize consecutive encoded characters
inputs  = np.array([encoded_text[i : i + windowSize]         for i in range(num_sequences)], dtype=np.int32)
targets = np.array([encoded_text[i + 1 : i + windowSize + 1] for i in range(num_sequences)], dtype=np.int32)

# Print shapes
print(f"Inputs  shape: {inputs.shape}")
print(f"Targets shape: {targets.shape}")

# Print first row to verify shift
print(f"\nFirst row of inputs:  {inputs[0]}")
print(f"First row of targets: {targets[0]}")
print("\nVisual check (targets should be inputs shifted by 1):")
print(f"  inputs[0][1:6]  = {inputs[0][1:6]}")
print(f"  targets[0][0:5] = {targets[0][0:5]}")

Inputs  shape: (841282, 128)
Targets shape: (841282, 128)

First row of inputs:  [ 6  6  6  1 41 42 23 40 42  1 37 28  1 42 30 27  1 38 40 37 32 27 25 42
  1 29 43 42 27 36 24 27 40 29  1 27 24 37 37 33  1 13 14 15  1  6  6  6
  0  0  0  0  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1 26 40 23 25 43 34 23  0  0  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1]
First row of targets: [ 6  6  1 41 42 23 40 42  1 37 28  1 42 30 27  1 38 40 37 32 27 25 42  1
 29 43 42 27 36 24 27 40 29  1 27 24 37 37 33  1 13 14 15  1  6  6  6  0
  0  0  0  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1 26 40 23 25 43 34 23  0  0  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1 51]

Visual check (targets should be inputs shifted by 1):
  inputs[0][1:6]  = [ 6  6  1 41 42]
  targets[0][0:5] = [ 6

Task-5

In [21]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype=tf.bool):
    """
    Creates a causal (lower-triangular) attention mask.
    Allowed positions (past + current) -> True
    Future positions -> False

    Args:
        batch_size: number of samples in batch
        n_dest:     number of destination (query) positions
        n_src:      number of source (key/value) positions
        dtype:      output dtype
    Returns:
        Boolean mask of shape (batch_size, 1, n_dest, n_src)
    """
    i = tf.range(n_dest)[:, None]   # shape (n_dest, 1)
    j = tf.range(n_src)[None, :]    # shape (1, n_src)
    # True where j <= i  (allow attending to current and past positions)
    mask = tf.cast(j <= i, dtype=dtype)  # shape (n_dest, n_src)
    # Expand for batch and head dimensions -> (batch_size, 1, n_dest, n_src)
    mask = tf.reshape(mask, [1, 1, n_dest, n_src])
    mask = tf.tile(mask, [batch_size, 1, 1, 1])
    return mask

# Quick test of the mask
test_mask = causal_attention_mask(1, 4, 4)
print("Causal mask (4x4, batch=1):")
print(test_mask[0, 0].numpy())

Causal mask (4x4, batch=1):
[[ True False False False]
 [ True  True False False]
 [ True  True  True False]
 [ True  True  True  True]]


Task-6

In [22]:
class TransformerBlock(layers.Layer):
    """
    A single Transformer decoder block with:
      - Causal MultiHeadAttention
      - Feed-forward network (FFN)
      - Two LayerNormalization layers
      - Two Dropout layers
    """

    def __init__(self, num_heads, dim_model, ff_dim, dropout_rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.num_heads   = num_heads
        self.dim_model   = dim_model
        self.ff_dim      = ff_dim
        self.dropout_rate = dropout_rate

        # 1. Multi-Head Self-Attention
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=dim_model // num_heads
        )

        # 2. Feed-Forward Network
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(dim_model)
        ])

        # 3. Layer Normalization layers
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        # 4. Dropout layers
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        """
        Forward pass:
          attention -> optional dropout -> residual add -> normalization
          -> FFN -> optional dropout -> residual add -> normalization
        """
        # Determine sequence length and batch size from input
        input_shape = tf.shape(x)
        batch_size  = input_shape[0]
        seq_len     = input_shape[1]

        # Build the causal mask: shape (batch_size, 1, seq_len, seq_len)
        mask = causal_attention_mask(batch_size, seq_len, seq_len, dtype=tf.bool)

        # Self-attention with causal mask
        # training=False is passed to MHA to disable its internal dropout during generation
        attn_output = self.attention(
            query=x, value=x, key=x,
            attention_mask=mask,
            training=training
        )

        # Dropout after attention
        attn_output = self.dropout1(attn_output, training=training)

        # Residual connection + Layer Norm 1
        out1 = self.layernorm1(x + attn_output)

        # Feed-forward network
        ffn_output = self.ffn(out1)

        # Dropout after FFN
        ffn_output = self.dropout2(ffn_output, training=training)

        # Residual connection + Layer Norm 2
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

    def get_config(self):
        config = super().get_config()
        config.update({
            'num_heads':    self.num_heads,
            'dim_model':    self.dim_model,
            'ff_dim':       self.ff_dim,
            'dropout_rate': self.dropout_rate
        })
        return config

print("TransformerBlock class defined successfully.")

TransformerBlock class defined successfully.


Task 7, build tiny character model: 35 points

In [23]:
class TinyCharModel(tf.keras.Model):
    """
    Tiny Transformer-based character-level language model.
    Architecture:
      Token Embedding + Positional Embedding
      -> numLayers x TransformerBlock
      -> LayerNormalization
      -> Dense(vocabSize)  [raw logits, NO softmax]
    """

    def __init__(self, vocab_size, max_length, dim_model,
                 num_layers, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super(TinyCharModel, self).__init__(**kwargs)

        self.vocab_size   = vocab_size
        self.max_length   = max_length
        self.dim_model    = dim_model
        self.num_layers   = num_layers
        self.num_heads    = num_heads
        self.ff_dim       = ff_dim
        self.dropout_rate = dropout_rate

        # 1. Token embedding: (vocabSize x dimModel)
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=dim_model
        )

        # 2. Positional embedding: (maxLength x dimModel)
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=dim_model
        )

        # 3. Stack of Transformer blocks
        self.transformer_blocks = [
            TransformerBlock(num_heads, dim_model, ff_dim, dropout_rate)
            for _ in range(num_layers)
        ]

        # 4. Final LayerNorm
        self.final_layernorm = layers.LayerNormalization(epsilon=1e-6)

        # 5. Output Dense layer - raw logits, NO softmax
        self.output_dense = layers.Dense(vocab_size)

        # Optional dropout after embedding sum
        self.embedding_dropout = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        """
        Forward pass.
        x: integer token indices of shape (batch_size, seq_len)
        """
        seq_len = tf.shape(x)[1]

        # Create position indices: [0, 1, 2, ..., seq_len-1]
        positions = tf.range(start=0, limit=seq_len, delta=1)

        # Token + positional embeddings
        token_emb = self.token_embedding(x)           # (batch, seq_len, dim_model)
        pos_emb   = self.position_embedding(positions) # (seq_len, dim_model)

        # Combine embeddings
        x = token_emb + pos_emb
        x = self.embedding_dropout(x, training=training)

        # Pass through each Transformer block
        for block in self.transformer_blocks:
            x = block(x, training=training)

        # Final normalization
        x = self.final_layernorm(x)

        # Output logits (no softmax)
        logits = self.output_dense(x)  # (batch, seq_len, vocab_size)

        return logits

    def get_config(self):
        config = super().get_config()
        config.update({
            'vocab_size':   self.vocab_size,
            'max_length':   self.max_length,
            'dim_model':    self.dim_model,
            'num_layers':   self.num_layers,
            'num_heads':    self.num_heads,
            'ff_dim':       self.ff_dim,
            'dropout_rate': self.dropout_rate
        })
        return config

# Instantiate the model
tinyModel = TinyCharModel(
    vocab_size=vocabSize,
    max_length=maxLength,
    dim_model=dimModel,
    num_layers=numLayers,
    num_heads=numHeads,
    ff_dim=ffDim
)

print("TinyCharModel instantiated successfully.")

TinyCharModel instantiated successfully.


Task 8, Compile and Summarize model: 15 points

In [24]:
# Compile the model
tinyModel.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

# BUILD the model by calling it once (if needed, depends on how you built the model)
# Uncomment this code only if you see the model summary below, but it shows
# that there are 0 trainable parameters.

buildVar = tf.zeros((1, windowSize), dtype=tf.int32)
dummyVar = tinyModel(buildVar)

tinyModel.summary()

Model: "tiny_char_model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (1, 128, 128)          │        10,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (128, 128)             │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ ?                      │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ ?                      │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_9           │ (1, 128, 128)          │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (1, 128, 80)           │        10,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ ?                      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 302,288 (1.15 MB)

 Trainable params: 302,288 (1.15 MB)

 Non-trainable params: 0 (0.00 B)

Task 9, Train the model with at least 2 epochs: 10 points

In [25]:
history = tinyModel.fit(
    inputs,
    targets,
    batch_size=batchSize,
    epochs=epochs,
    verbose=1
)

print("\nTraining complete.")
print(f"Final loss after {epochs} epochs: {history.history['loss'][-1]:.4f}")

Epoch 1/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 191s 14ms/step - loss: 1.6027
Epoch 2/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 174s 13ms/step - loss: 1.4349
Epoch 3/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 173s 13ms/step - loss: 1.3950
Epoch 4/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 174s 13ms/step - loss: 1.3718
Epoch 5/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 175s 13ms/step - loss: 1.3565
Epoch 6/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 174s 13ms/step - loss: 1.3452
Epoch 7/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 174s 13ms/step - loss: 1.3359
Epoch 8/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 174s 13ms/step - loss: 1.3287
Epoch 9/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 173s 13ms/step - loss: 1.3229
Epoch 10/10
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 174s 13ms/step - loss: 1.3184

Training complete.
Final loss after 10 epochs: 1.3184


Task-10 : Return character ID to generate: 20 points

In [26]:
def sample_next_char_id(logits, temperature=1.0, top_k=40):
    """
    Given logits for a single time step, apply:
      1. Temperature scaling
      2. Top-k filtering (keep only top k tokens)
      3. Softmax to get probabilities
      4. Sample one token

    Args:
        logits:      1D array/tensor of raw logits, shape (vocab_size,)
        temperature: float > 0. Lower -> more deterministic; higher -> more random.
        top_k:       int. Keep only the top_k highest-probability tokens.

    Returns:
        int: the sampled character ID
    """
    # Convert to numpy float64 for numerical stability
    logits = np.array(logits, dtype=np.float64)

    # 1. Temperature scaling
    logits = logits / temperature

    # 2. Top-k filtering
    if top_k > 0 and top_k < len(logits):
        # Find the k-th largest value (threshold)
        top_k_indices = np.argsort(logits)[-top_k:]  # indices of top-k logits
        # Set all other logits to a very large negative number
        mask = np.full_like(logits, fill_value=-1e10)
        mask[top_k_indices] = logits[top_k_indices]
        logits = mask

    # 3. Softmax for numerical stability: subtract max first
    logits -= np.max(logits)
    probs = np.exp(logits)
    probs /= np.sum(probs)

    # 4. Sample one character ID from the distribution
    char_id = np.random.choice(len(probs), p=probs)
    return int(char_id)

print("sample_next_char_id function defined successfully.")

sample_next_char_id function defined successfully.


Task 11 generate output text from seed: 15 points

In [27]:
def generate(model, seed_text, num_chars=250, temperature=0.8, top_k=40):
    """
    Generate text character-by-character from a seed string.

    Args:
        model:       the trained TinyCharModel
        seed_text:   string to use as starting context
        num_chars:   number of NEW characters to generate (>= 250)
        temperature: controls randomness (lower = more deterministic)
        top_k:       top-k sampling parameter

    Returns:
        str: seed_text + generated characters
    """
    # Start with the seed encoded as a list of integers
    generated_ids = list(encode(seed_text))

    for _ in range(num_chars):
        # Take the last windowSize characters as context (pad if shorter)
        context = generated_ids[-windowSize:]

        # Pad from left if context is shorter than windowSize
        if len(context) < windowSize:
            context = [0] * (windowSize - len(context)) + context

        # Build input tensor of shape (1, windowSize)
        input_tensor = np.array([context], dtype=np.int32)

        # Forward pass: get logits of shape (1, windowSize, vocabSize)
        logits = model(input_tensor, training=False)

        # Extract logits for the LAST time step: shape (vocabSize,)
        last_logits = logits[0, -1, :].numpy()

        # Sample the next character ID
        next_id = sample_next_char_id(last_logits, temperature=temperature, top_k=top_k)

        # Append to generated sequence
        generated_ids.append(next_id)

    # Decode full sequence back to string
    full_text = decode(generated_ids)
    return full_text


# Generate at least 250 characters from a seed
seed = "Then Count Dracula said to me:"
print(f"Generating text with seed: '{seed}'")
print("-" * 60)

output_text = generate(
    model=tinyModel,
    seed_text=seed,
    num_chars=250,
    temperature=0.8,
    top_k=40
)

print(output_text)

Generating text with seed: 'Then Count Dracula said to me:'
------------------------------------------------------------
Then Count Dracula said to me:
Mrs.
Ah!
So


CHAPTER


CHAPTER XIII

MINA



CHAPTER XXVIII

DR. SEWARDS DIARY



_3 SEWARDS DIARY


_19 October._--On 13 September._

My death--I suppose is to me. I am getting in Mrs. Harker.

       THE OLSENLABME MANA HARKER.


_Letter, Madam M
